# mini-gpt — Kaggle Olsson-approximate 2-layer experiment

**Goal**: reproduce the induction-head phase change described in Olsson et al. (2022).

Config: n_layers=2, d_model=512, d_k=64, batch=64, 120K steps → 1.97B tokens.

**Usage**: Upload to Kaggle → Settings: GPU T4 x2 or P100, Internet ON → Save Version (commit mode).

Run cells top-to-bottom. For resume after session reset, skip to Cell 9.

In [ ]:
# Cell 1: GPU check
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Install dependencies
!pip install tokenizers wandb -q

In [ ]:
# Cell 3: Clone or pull repo
import os
if not os.path.exists('/kaggle/working/mini-gpt'):
    !git clone https://github.com/daisukeabe32/mini-gpt /kaggle/working/mini-gpt -q
    print('cloned')
else:
    !git -C /kaggle/working/mini-gpt pull -q
    print('pulled latest')
%cd /kaggle/working/mini-gpt
!git log --oneline -3

In [ ]:
# Cell 4: W&B login via Kaggle Secret
# Add secret key 'WANDB_API_KEY' in Kaggle notebook Settings > Secrets, then enable 'Attach to notebook'.
import os
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
import wandb
wandb.login()
print('W&B ready')

In [ ]:
# Cell 5: Copy tokenized data from Kaggle Dataset -> working dir (once per session, ~2-3 min)
import os, shutil
SRC = '/kaggle/input/datasets/da3246/mini-gpt-tinystories/bpe_hf_30000_tinystories'
DST = 'data/tokenized/bpe_hf_30000_tinystories'
os.makedirs(DST, exist_ok=True)
for fname in ['tokenizer.json', 'hf_tokenizer.json', 'val_ids.pt', 'train_ids.pt']:
    dst_path = f'{DST}/{fname}'
    if not os.path.exists(dst_path):
        print(f'copying {fname} ...')
        shutil.copy2(f'{SRC}/{fname}', dst_path)
        print(f'  done ({os.path.getsize(dst_path)/1e6:.0f} MB)')
    else:
        print(f'  {fname} — skip (exists)')

In [ ]:
# Cell 6: Verify tokenized data
import torch
train = torch.load('data/tokenized/bpe_hf_30000_tinystories/train_ids.pt')
val   = torch.load('data/tokenized/bpe_hf_30000_tinystories/val_ids.pt')
print(f'train: {len(train):,} tokens')
print(f'val  : {len(val):,} tokens')

In [ ]:
# Cell 7: Prepare checkpoint output directory
import os
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CKPT_DIR}')
print('NOTE: in Save Version (commit mode) this directory is persisted as Kaggle Output.')

In [ ]:
# Cell 8: START TRAINING — Olsson-approximate 2-layer config
#
# Design intent:
#   n_layers=2     → Olsson's minimal setup where induction heads are cleanest
#   d_model=512    → 67% of Olsson's 768; big enough for circuits to form
#   d_k=64         → exact match with Olsson
#   num_heads=8    → d_model / d_k
#   d_ff=2048      → 4x d_model (standard transformer ratio)
#   batch_size=64  → larger than previous run; more tokens per step
#   max_iters=120K → 64 * 256 * 120000 = 1.97B tokens  (> Olsson phase-change lower bound 1.5B)
#   save_every=2000 → snapshots every 64*256*2000 = 32.8M tokens for emergence curve
#
# Total parameters: ~25M (vs Olsson's 4-layer 8-head 512-dim run at ~44M)
#
import os
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

!python -u -m scripts.train_char_gpt \
    --tokenized_dir  data/tokenized/bpe_hf_30000_tinystories \
    --block_size  256 \
    --d_model     512 \
    --n_layers      2 \
    --num_heads     8 \
    --d_k          64 \
    --d_ff       2048 \
    --batch_size   64 \
    --max_iters  120000 \
    --lr          3e-4 \
    --min_lr      3e-5 \
    --warmup_iters 1000 \
    --eval_every  1000 \
    --save_every  2000 \
    --checkpoint_dir {CKPT_DIR} \
    2>&1 | tee {CKPT_DIR}/training_log.txt

In [ ]:
# Cell 9: RESUME TRAINING — after session reset
# 1. Find previous run Output in Kaggle (your notebook > Output tab > checkpoints/)
# 2. Add that Output as a dataset input: Edit notebook > Add Data > Your Work
# 3. Replace RUN_ID below with the folder name under /kaggle/input/
import os
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Show available inputs to find the right path
print('Available inputs:')
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt'):
            print(f'  {os.path.join(root, f)}')

In [ ]:
# Cell 10: RESUME — set RESUME_PT from Cell 9 output, then run
import os
CKPT_DIR  = '/kaggle/working/checkpoints'
RESUME_PT = '/kaggle/input/XXXXX/checkpoints/YYYYMMDD_HHMMSS/best.pt'  # <-- replace
print(f'Resuming from: {RESUME_PT}')
print(f'Exists: {os.path.exists(RESUME_PT)}')

!python -u -m scripts.train_char_gpt \
    --tokenized_dir  data/tokenized/bpe_hf_30000_tinystories \
    --block_size  256 \
    --d_model     512 \
    --n_layers      2 \
    --num_heads     8 \
    --d_k          64 \
    --d_ff       2048 \
    --batch_size   64 \
    --max_iters  120000 \
    --lr          3e-4 \
    --min_lr      3e-5 \
    --warmup_iters 1000 \
    --eval_every  1000 \
    --save_every  2000 \
    --checkpoint_dir {CKPT_DIR} \
    --resume_from {RESUME_PT} \
    2>&1 | tee {CKPT_DIR}/training_log_resumed.txt

In [ ]:
# Cell 11: Check training progress
import os
CKPT_DIR = '/kaggle/working/checkpoints'
for log_name in ['training_log.txt', 'training_log_resumed.txt']:
    log_path = f'{CKPT_DIR}/{log_name}'
    if not os.path.exists(log_path):
        continue
    with open(log_path) as f:
        lines = [l for l in f if l.startswith('step')]
    print(f'=== {log_name} ({len(lines)} evals) ===')
    if lines:
        print('first :', lines[0].strip())
        print('latest:', lines[-1].strip())

In [ ]:
# Cell 12: Induction head analysis — run after training completes
#
# analyze_attention.py runs the Olsson-style test:
#   1. Build a repeated random sequence [t0..tN t0..tN]
#   2. Extract per-head attention weights and value outputs
#   3. Compute prefix-matching score (attention-weight diagonal) and
#      copying score (logit-lens direct path z_h -> W_O_h -> W_U)
#   4. Saves attention heatmap grid to figs/attention_grid.png
#
import os, glob
CKPT_DIR = '/kaggle/working/checkpoints'

# Find the best.pt in the most recent run subfolder
runs = sorted(glob.glob(f'{CKPT_DIR}/*/best.pt'))
if not runs:
    print('No best.pt found — training may not have finished yet')
else:
    CKPT_PT = runs[-1]
    print(f'Using checkpoint: {CKPT_PT}')
    os.makedirs('figs', exist_ok=True)
    !python -m scripts.analyze_attention \
        --checkpoint {CKPT_PT} \
        --seq_len 64 \
        --out_dir figs/

In [ ]:
# Cell 13: Emergence curve — induction scores across snapshots
#
# Scans every snapshot checkpoint saved during training, computes the max
# induction score (prefix-matching) across all heads, and plots it vs tokens seen.
# This reproduces Olsson's Figure 1 emergence curve.
#
import os, glob, torch
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/kaggle/working/mini-gpt')

from scripts.analyze_attention import load_model, make_repeated_sequence, extract_attention, induction_score

CKPT_DIR = '/kaggle/working/checkpoints'
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN  = 64

# Collect all step-XXXXX checkpoints (not best.pt)
snap_paths = sorted(glob.glob(f'{CKPT_DIR}/*/step_*.pt'))
print(f'Found {len(snap_paths)} snapshots')

results = []  # (tokens_seen, max_induction_score)
for path in snap_paths:
    try:
        model, tok, cfg = load_model(path, device)
        ckpt  = torch.load(path, map_location=device)
        step  = ckpt['step']
        tokens = step * cfg['batch_size'] * cfg['block_size']  # approximate
        ids, _ = make_repeated_sequence(tok, SEQ_LEN, device)
        all_weights, _ = extract_attention(model, ids)
        max_score = max(
            induction_score(all_weights[l][:, h], SEQ_LEN)
            for l in range(cfg['n_layers'])
            for h in range(cfg['num_heads'])
        )
        results.append((tokens, max_score))
        print(f'  step={step:>7,} | tokens={tokens/1e9:.3f}B | max_induction={max_score:.4f}')
    except Exception as e:
        print(f'  skip {path}: {e}')

if results:
    xs, ys = zip(*sorted(results))
    plt.figure(figsize=(8, 4))
    plt.plot([x/1e9 for x in xs], ys, marker='o', ms=4)
    plt.xlabel('Tokens seen (B)')
    plt.ylabel('Max induction score')
    plt.title('Induction head emergence curve')
    plt.grid(True, alpha=0.3)
    os.makedirs('figs', exist_ok=True)
    plt.savefig('figs/emergence_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved figs/emergence_curve.png')